# HumanEval Benchmark — Gemma 4 31B (Google AI Studio)
**Reproducing & Extending the HumanEval Paper (Chen et al., 2021)**

Testing Gemma 4 31B — released April 2, 2026 — on the original 164-problem HumanEval benchmark.

---

## CELL 1 — API key



In [ ]:

API_KEY = "YOUR_GOOGLE_AI_STUDIO_KEY_HERE"  # Get free key at: aistudio.google.com/apikey

# Model — Gemma 4 31B via Google AI Studio
# (Google AI Studio uses gemma-4 model name)
MODEL = "models/gemma-4-31b-it"  # can update this in cell 4 test

print("Config loaded!")
print("API key set:", API_KEY[:10] + "...")

## CELL 2 — Install Dependencies

In [ ]:
!pip install google-genai datasets requests human-eval --quiet
print("Done!")

## CELL 3 — Check Which Gemma 4 Models Are Available

In [ ]:
from google import genai

client = genai.Client(api_key=API_KEY)

# List available models
print("Available Gemma models:")
print("-" * 40)
for model in client.models.list():
    if 'gemma' in model.name.lower():
        print(model.name)

## CELL 4 — Set the Correct Model Name + API Test

In [ ]:
from google import genai
from google.genai import types
from datasets import load_dataset
import time

client = genai.Client(api_key=API_KEY)
MODEL = "models/gemma-4-31b-it"

def query_model(prompt, max_retries=3):
    system_instruction = """You are a Python coding assistant.
Complete the function body ONLY.
Return ONLY raw Python code, no explanation, no markdown, no backticks.
Start directly with the indented code."""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=f"Complete this Python function:\n\n{prompt}",
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.2,
                    max_output_tokens=512,
                )
            )
            return response.text
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(10)
    return None

# Load dataset
dataset = load_dataset("openai/openai_humaneval", split="test")

# API test with 1 problem
print("Testing API...")
print("-" * 50)
result = query_model(dataset[0]['prompt'])
print(result)

## CELL 5 — Run Full Benchmark (All 164 Problems)

In [ ]:
import json, time, os
from google.colab import drive

# 1. Setup paths
LOCAL_PATH = "/content/gemma4_results.jsonl"
DRIVE_PATH = "/content/drive/MyDrive/gemma4_results.jsonl"

# 2. Try to bring back your progress from Drive if local is empty
if not os.path.exists(LOCAL_PATH):
    try:
        drive.mount('/content/drive', force_remount=True)
        if os.path.exists(DRIVE_PATH):
            import shutil
            shutil.copy(DRIVE_PATH, LOCAL_PATH)
            print("Successfully recovered progress from Google Drive!")
    except:
        print("Starting from scratch (Drive not connected or file missing).")

# 3. Identify what is already done
completed_ids = set()
if os.path.exists(LOCAL_PATH):
    with open(LOCAL_PATH, "r") as f:
        for line in f:
            try:
                data = json.loads(line)
                completed_ids.add(data["task_id"])
            except: continue

print(f"Resuming benchmark: {len(completed_ids)} problems already finished.")
print(f"Starting Gemma 4 31B on remaining {len(dataset) - len(completed_ids)} problems")
print("-" * 50)

# 4. Run the loop (saving LOCALLY to avoid Transport Errors)
with open(LOCAL_PATH, "a") as f:
    for i, problem in enumerate(dataset):
        task_id = problem['task_id']

        if task_id in completed_ids:
            continue

        completion = query_model(problem['prompt'])
        result = {
            "task_id": task_id,
            "completion": completion if completion else "    pass"
        }

        f.write(json.dumps(result) + "\n")
        f.flush()

        status = "✅" if completion else "❌"
        print(f"  {status} {len(completed_ids) + 1}/164 — {task_id}")
        completed_ids.add(task_id)

        time.sleep(4)

# 5. Final backup to Drive once 100% finished
try:
    import shutil
    shutil.copy(LOCAL_PATH, DRIVE_PATH)
    print("-" * 50)
    print("SUCCESS: Full benchmark saved to Google Drive!")
except Exception as e:
    print(f"Warning: Could not backup to Drive: {e}")
    print("Manually download 'gemma4_results.jsonl' from the folder icon on the left!")

## CELL 6 — Evaluate Results (Calculate pass@1)

In [ ]:
# Run the official HumanEval evaluator
!evaluate_functional_correctness gemma4_results.jsonl

print("\nLook above for your pass@1 score!")

## CELL 7 — Make the Comparison Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ============================================================
# After Cell 6 runs, look at the pass@1 score in the output
# and replace the 0.0 below with your actual score (as %)
# e.g. if it says pass@1 = 0.756, enter 75.6
# ============================================================

models    = ["Codex\n(2021 Paper)", "GPT-3.5", "GPT-4", "Gemma 4 31B\n(Our Test)"]
scores    = [28.8,                   48.1,       67.0,    95.1]  # <-- replace last 0.0
our_test  = [False,                  False,      False,   True]

colors = ["#94a3b8" if not o else "#4f46e5" for o, m in zip(our_test, models)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models, scores, color=colors, edgecolor="white", linewidth=1.5, width=0.5)

for bar, val in zip(bars, scores):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val}%", ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_ylabel("pass@1 Score (%)", fontsize=12)
ax.set_title("HumanEval Benchmark — Historical vs Gemma 4 31B (April 2026)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylim(0, 105)
ax.set_facecolor("#f8fafc")
fig.patch.set_facecolor("#ffffff")
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

legend_patches = [
    mpatches.Patch(color="#94a3b8", label="Historical results (published)"),
    mpatches.Patch(color="#4f46e5", label="Gemma 4 31B — our independent test"),
]
ax.legend(handles=legend_patches, loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.savefig("humaneval_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as humaneval_chart.png")

## CELL 8 — Failure Analysis

In [ ]:
import json

# Load the scored results (evaluator adds 'passed' field)
try:
    scored = {}
    with open("gemma4_results.jsonl_results.jsonl") as f:
        for line in f:
            r = json.loads(line)
            scored[r['task_id']] = r

    passed = [k for k, v in scored.items() if v.get('passed')]
    failed = [k for k, v in scored.items() if not v.get('passed')]

    print("=" * 50)
    print("RESULTS BREAKDOWN")
    print("=" * 50)
    print(f"Total problems:  164")
    print(f"Passed:          {len(passed)}")
    print(f"Failed:          {len(failed)}")
    print(f"pass@1 score:    {len(passed)/164*100:.1f}%")
    print()
    print("Failed problems:")
    for t in sorted(failed):
        print(f"  - {t}")

except FileNotFoundError:
    print("Run Cell 6 first to generate scored results.")

## CELL 9 — Final Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║         HUMANEVAL BENCHMARK — FINAL RESULTS              ║
╠══════════════════════════════════════════════════════════╣
║  Model                  │ Year  │ pass@1  │ Source       ║
╠══════════════════════════════════════════════════════════╣
║  Codex (code-davinci)   │ 2021  │  28.8%  │ Paper        ║
║  GPT-3.5                │ 2022  │  48.1%  │ Published    ║
║  GPT-4                  │ 2023  │  67.0%  │ Published    ║
║  Gemma 4 31B            │ 2026  │  95.1%  │ This Study   ║
╚══════════════════════════════════════════════════════════╝
""")

In [ ]:
import os
with open(SAVE_PATH) as f:
    lines = f.readlines()
print(f"Saved so far: {len(lines)}/164")